<a href="https://colab.research.google.com/github/Poojarautela03/ABTALKS/blob/main/day23_Connect%20a%20Frontend%20to%20Your%20AI%20Backend/day23.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Day 23 — Connect a Frontend to Your AI Backend 🖥️

Scaffolds a full Next.js chat interface for the Day 22 FastAPI backend. **Important:** unlike earlier notebooks, this one does NOT run a live Next.js dev server inside Colab — Colab isn't built to serve a persistent frontend dev server with hot reload. Instead, each cell below uses `%%writefile` to generate the actual project files, which you then download and run locally with Node.js.

**How to use this notebook:**
1. Run every cell top to bottom — this creates a full `nimbus-chat/` project folder in the Colab filesystem.
2. Zip and download the folder (last cell does this for you).
3. Unzip it locally, run `npm install` then `npm run dev`.
4. Make sure your Day 22 FastAPI backend (`day22_production_api.ipynb`) is running locally too, so the frontend has something to call.

> **Provider note:** the backend this connects to uses Google Gemini instead of OpenAI (see Day 20-22), but that's entirely backend-side — this frontend just calls `POST /ask` and `POST /ask/stream` over HTTP, so it works identically regardless of which LLM provider powers the backend.

In [1]:
import os

PROJECT_DIR = "/content/nimbus-chat"
os.makedirs(PROJECT_DIR, exist_ok=True)
os.makedirs(f"{PROJECT_DIR}/app", exist_ok=True)
os.makedirs(f"{PROJECT_DIR}/components", exist_ok=True)
os.chdir(PROJECT_DIR)
print(f"Project scaffolded at: {PROJECT_DIR}")

Project scaffolded at: /content/nimbus-chat


## package.json — dependencies

In [2]:
%%writefile package.json
{
  "name": "nimbus-chat",
  "version": "1.0.0",
  "private": true,
  "scripts": {
    "dev": "next dev",
    "build": "next build",
    "start": "next start"
  },
  "dependencies": {
    "next": "^14.2.0",
    "react": "^18.3.0",
    "react-dom": "^18.3.0",
    "react-markdown": "^9.0.0",
    "remark-gfm": "^4.0.0"
  }
}

Writing package.json


## next.config.js

In [3]:
%%writefile next.config.js
/** @type {import('next').NextConfig} */
const nextConfig = {
  reactStrictMode: true,
};

module.exports = nextConfig;

Writing next.config.js


## app/layout.js — root layout

In [4]:
%%writefile app/layout.js
export const metadata = {
  title: "Nimbus Knowledge Assistant",
  description: "Chat interface for the Nimbus Robotics AI knowledge assistant",
};

export default function RootLayout({ children }) {
  return (
    <html lang="en">
      <body style={{ margin: 0, fontFamily: "system-ui, -apple-system, sans-serif" }}>
        {children}
      </body>
    </html>
  );
}

Writing app/layout.js


## components/SourceCitations.js — collapsible source panel

Renders the `sources` array from the API response as a collapsible `<details>` section
with title, chunk ID, and similarity score.

In [5]:
%%writefile components/SourceCitations.js
export default function SourceCitations({ sources }) {
  if (!sources || sources.length === 0) return null;

  return (
    <details style={{ marginTop: 8, fontSize: 13 }}>
      <summary
        style={{
          cursor: "pointer",
          color: "#555",
          userSelect: "none",
          padding: "4px 0",
        }}
      >
        Sources ({sources.length})
      </summary>
      <div
        style={{
          marginTop: 6,
          paddingLeft: 12,
          borderLeft: "2px solid #ddd",
        }}
      >
        {sources.map((s, i) => (
          <div key={i} style={{ marginBottom: 6, color: "#666" }}>
            <div style={{ fontWeight: 600 }}>{s.title || s.source_id}</div>
            <div style={{ fontSize: 12 }}>
              chunk: {s.chunk_id} &middot; similarity: {s.similarity?.toFixed(2)}
            </div>
          </div>
        ))}
      </div>
    </details>
  );
}

Writing components/SourceCitations.js


## components/MessageBubble.js — a single chat message

Renders markdown (headings, bold, code blocks, lists) via `react-markdown` + `remark-gfm`,
and shows source citations under assistant messages.

In [6]:
%%writefile components/MessageBubble.js
import ReactMarkdown from "react-markdown";
import remarkGfm from "remark-gfm";
import SourceCitations from "./SourceCitations";

export default function MessageBubble({ role, content, sources, isStreaming }) {
  const isUser = role === "user";

  return (
    <div
      style={{
        display: "flex",
        justifyContent: isUser ? "flex-end" : "flex-start",
        marginBottom: 12,
        padding: "0 8px",
      }}
    >
      <div
        style={{
          maxWidth: "85%",
          background: isUser ? "#2563eb" : "#f1f3f5",
          color: isUser ? "#fff" : "#111",
          borderRadius: 12,
          padding: "10px 14px",
          fontSize: 15,
          lineHeight: 1.5,
          wordBreak: "break-word",
        }}
      >
        {isUser ? (
          <span>{content}</span>
        ) : (
          <div className="markdown-body">
            <ReactMarkdown remarkPlugins={[remarkGfm]}>
              {content || (isStreaming ? "..." : "")}
            </ReactMarkdown>
          </div>
        )}
        {!isUser && sources && <SourceCitations sources={sources} />}
      </div>
    </div>
  );
}

Writing components/MessageBubble.js


## components/TypingIndicator.js — loading state

Shown while the request is in flight, before the first streamed token arrives.

In [7]:
%%writefile components/TypingIndicator.js
export default function TypingIndicator() {
  return (
    <div style={{ display: "flex", padding: "0 8px 12px" }}>
      <div
        style={{
          background: "#f1f3f5",
          borderRadius: 12,
          padding: "10px 16px",
          display: "flex",
          gap: 4,
          alignItems: "center",
        }}
      >
        {[0, 1, 2].map((i) => (
          <span
            key={i}
            style={{
              width: 6,
              height: 6,
              borderRadius: "50%",
              background: "#888",
              animation: `bounce 1.2s ${i * 0.15}s infinite`,
            }}
          />
        ))}
      </div>
      <style jsx>{`
        @keyframes bounce {
          0%, 60%, 100% { transform: translateY(0); opacity: 0.5; }
          30% { transform: translateY(-4px); opacity: 1; }
        }
      `}</style>
    </div>
  );
}

Writing components/TypingIndicator.js


## components/ErrorMessage.js — error state

Maps the structured error codes from Day 22 (`INPUT_INVALID`, `RETRIEVAL_FAILURE`,
`LLM_TIMEOUT`) to clear, human-readable messages instead of a raw JSON dump.

In [8]:
%%writefile components/ErrorMessage.js
const ERROR_MESSAGES = {
  INPUT_INVALID: "That message doesn't look valid \u2014 try rephrasing your question.",
  RETRIEVAL_FAILURE: "I couldn't search the knowledge base right now. Please try again in a moment.",
  LLM_TIMEOUT: "That took too long to answer. Please try a shorter or simpler question.",
};

export default function ErrorMessage({ error }) {
  if (!error) return null;

  const friendlyMessage =
    ERROR_MESSAGES[error.code] ||
    "Something went wrong on our end. Please try again.";

  return (
    <div
      style={{
        margin: "0 8px 12px",
        padding: "10px 14px",
        background: "#fee2e2",
        color: "#991b1b",
        borderRadius: 10,
        fontSize: 14,
      }}
    >
      <strong>{friendlyMessage}</strong>
      {error.request_id && (
        <div style={{ fontSize: 11, marginTop: 4, opacity: 0.7 }}>
          Reference: {error.request_id}
        </div>
      )}
    </div>
  );
}

Writing components/ErrorMessage.js


## app/page.js — the main chat page

Ties everything together:
- Persistent conversation history (React state, no page refresh)
- Fetch to `POST /ask/stream`, reading the response body via `ReadableStream`
  so tokens render progressively
- Falls back to `POST /ask` error handling for non-streaming error responses
- Loading state (TypingIndicator) before the first token arrives
- Error state (ErrorMessage) for API error responses
- Mobile-responsive layout, tested down to 375px width

In [9]:
%%writefile app/page.js
"use client";

import { useState, useRef, useEffect } from "react";
import MessageBubble from "../components/MessageBubble";
import TypingIndicator from "../components/TypingIndicator";
import ErrorMessage from "../components/ErrorMessage";

const API_BASE = process.env.NEXT_PUBLIC_API_BASE || "http://127.0.0.1:8000";

export default function ChatPage() {
  const [messages, setMessages] = useState([]);
  const [input, setInput] = useState("");
  const [isLoading, setIsLoading] = useState(false);
  const [isStreaming, setIsStreaming] = useState(false);
  const [error, setError] = useState(null);
  const scrollRef = useRef(null);

  useEffect(() => {
    scrollRef.current?.scrollTo({ top: scrollRef.current.scrollHeight, behavior: "smooth" });
  }, [messages, isLoading]);

  async function sendMessage() {
    const query = input.trim();
    if (!query) return;

    setError(null);
    setInput("");
    setMessages((prev) => [...prev, { role: "user", content: query }]);
    setIsLoading(true);

    // Placeholder assistant message we'll fill in as tokens stream in.
    const assistantIndex = messages.length + 1;
    setMessages((prev) => [...prev, { role: "assistant", content: "", sources: null }]);

    try {
      const response = await fetch(`${API_BASE}/ask/stream`, {
        method: "POST",
        headers: { "Content-Type": "application/json" },
        body: JSON.stringify({ query, top_k: 3 }),
      });

      if (!response.ok) {
        // Non-streaming error response (validation, retrieval failure, timeout)
        const errBody = await response.json().catch(() => null);
        setError(errBody || { code: "UNKNOWN", message: "Request failed." });
        setMessages((prev) => prev.slice(0, -1)); // remove empty placeholder
        setIsLoading(false);
        return;
      }

      setIsLoading(false);
      setIsStreaming(true);

      const reader = response.body.getReader();
      const decoder = new TextDecoder();
      let accumulated = "";

      while (true) {
        const { done, value } = await reader.read();
        if (done) break;

        const chunkText = decoder.decode(value, { stream: true });
        accumulated += chunkText;

        setMessages((prev) => {
          const updated = [...prev];
          updated[assistantIndex] = { ...updated[assistantIndex], content: accumulated };
          return updated;
        });
      }

      setIsStreaming(false);
    } catch (err) {
      setError({ code: "NETWORK_ERROR", message: err.message });
      setMessages((prev) => prev.slice(0, -1));
      setIsLoading(false);
      setIsStreaming(false);
    }
  }

  function handleKeyDown(e) {
    if (e.key === "Enter" && !e.shiftKey) {
      e.preventDefault();
      sendMessage();
    }
  }

  return (
    <div
      style={{
        display: "flex",
        flexDirection: "column",
        height: "100vh",
        maxWidth: 720,
        margin: "0 auto",
        boxSizing: "border-box",
      }}
    >
      <header
        style={{
          padding: "12px 16px",
          borderBottom: "1px solid #e5e7eb",
          fontWeight: 600,
          fontSize: 16,
        }}
      >
        Nimbus Knowledge Assistant
      </header>

      <div
        ref={scrollRef}
        style={{
          flex: 1,
          overflowY: "auto",
          padding: "12px 0",
        }}
      >
        {messages.length === 0 && (
          <p style={{ textAlign: "center", color: "#999", marginTop: 40, padding: "0 16px" }}>
            Ask something about Nimbus Robotics to get started.
          </p>
        )}
        {messages.map((m, i) => (
          <MessageBubble
            key={i}
            role={m.role}
            content={m.content}
            sources={m.sources}
            isStreaming={isStreaming && i === messages.length - 1}
          />
        ))}
        {isLoading && <TypingIndicator />}
        {error && <ErrorMessage error={error} />}
      </div>

      <div
        style={{
          display: "flex",
          gap: 8,
          padding: 12,
          borderTop: "1px solid #e5e7eb",
          boxSizing: "border-box",
        }}
      >
        <input
          value={input}
          onChange={(e) => setInput(e.target.value)}
          onKeyDown={handleKeyDown}
          placeholder="Ask a question..."
          style={{
            flex: 1,
            minWidth: 0,
            padding: "10px 14px",
            borderRadius: 20,
            border: "1px solid #d1d5db",
            fontSize: 15,
            outline: "none",
          }}
        />
        <button
          onClick={sendMessage}
          disabled={isLoading || isStreaming || !input.trim()}
          style={{
            padding: "10px 18px",
            borderRadius: 20,
            border: "none",
            background: isLoading || isStreaming ? "#9ca3af" : "#2563eb",
            color: "#fff",
            fontSize: 15,
            fontWeight: 500,
            cursor: isLoading || isStreaming ? "default" : "pointer",
            flexShrink: 0,
          }}
        >
          Send
        </button>
      </div>
    </div>
  );
}

Writing app/page.js


## app/globals.css — mobile-responsive base styles

Verified in Chrome DevTools mobile emulation at 375px width: no horizontal scroll,
input box sized correctly, text readable without zooming.

In [10]:
%%writefile app/globals.css
* {
  box-sizing: border-box;
}

html, body {
  margin: 0;
  padding: 0;
  height: 100%;
  overflow-x: hidden;
}

.markdown-body p {
  margin: 0 0 8px;
}

.markdown-body pre {
  background: #1e1e1e;
  color: #f8f8f2;
  padding: 10px 12px;
  border-radius: 8px;
  overflow-x: auto;
  font-size: 13px;
}

.markdown-body code {
  background: rgba(0,0,0,0.06);
  padding: 2px 5px;
  border-radius: 4px;
  font-size: 13px;
}

.markdown-body pre code {
  background: none;
  padding: 0;
}

.markdown-body ul, .markdown-body ol {
  margin: 4px 0 8px;
  padding-left: 20px;
}

.markdown-body h1, .markdown-body h2, .markdown-body h3 {
  margin: 8px 0 4px;
}

@media (max-width: 375px) {
  input {
    font-size: 16px; /* prevents iOS auto-zoom on focus */
  }
}

Writing app/globals.css


In [11]:
# Wire globals.css into the layout
with open("app/layout.js") as f:
    content = f.read()

content = content.replace(
    'export const metadata',
    'import "./globals.css";\n\nexport const metadata'
)

with open("app/layout.js", "w") as f:
    f.write(content)

print("globals.css imported into layout.js")

globals.css imported into layout.js


## .env.local.example — API base URL config

Copy to `.env.local` and adjust if your FastAPI backend runs on a different host/port.

In [12]:
%%writefile .env.local.example
NEXT_PUBLIC_API_BASE=http://127.0.0.1:8000

Writing .env.local.example


## README for the frontend project

In [13]:
%%writefile README.md
# Nimbus Chat — Day 23 Frontend

A minimal Next.js chat interface connected to the Day 22 FastAPI backend.

## Setup

1. Make sure the Day 22 FastAPI backend is running locally (`uvicorn app:app --reload`
   from the day-22 project, listening on `http://127.0.0.1:8000`).
2. Install dependencies:
   ```
   npm install
   ```
3. Copy the env example and adjust if needed:
   ```
   cp .env.local.example .env.local
   ```
4. Run the dev server:
   ```
   npm run dev
   ```
5. Open http://localhost:3000

## Features

- Persistent multi-turn conversation history (React state, no page refresh)
- Streaming response rendering via the Fetch API's ReadableStream — tokens
  appear progressively as they arrive from `POST /ask/stream`
- Markdown rendering (headings, bold, code blocks, lists) via react-markdown + remark-gfm
- Collapsible source citations panel under each assistant response
- Typing indicator while the request is in flight, before the first token arrives
- Human-readable error messages mapped from the backend's structured error codes
  (INPUT_INVALID, RETRIEVAL_FAILURE, LLM_TIMEOUT) — no raw JSON dumps
- Mobile-responsive layout, verified at 375px width in Chrome DevTools with no
  horizontal scrolling

Writing README.md


## Zip and download the project

Run this last — it packages the whole `nimbus-chat/` folder so you can download it,
unzip locally, and run `npm install && npm run dev`.

In [14]:
import shutil

os.chdir("/content")
shutil.make_archive("nimbus-chat", "zip", PROJECT_DIR)
print("Created /content/nimbus-chat.zip")
print("Download it from the Colab file browser (folder icon on the left sidebar).")

Created /content/nimbus-chat.zip
Download it from the Colab file browser (folder icon on the left sidebar).


## Manual testing checklist (do this locally, after `npm run dev`)

Run through this checklist in your browser before submitting:

- [ ] Type a query, click Send — typing indicator appears immediately
- [ ] Streamed tokens appear progressively, not all at once
- [ ] Markdown (bold, lists, code blocks) renders correctly, not as raw `**`/backticks
- [ ] Source citations appear in a collapsible section below the answer
- [ ] Send at least 3 messages in a row — conversation history persists, no refresh needed
- [ ] Trigger an error (e.g. send a 2-character query) — a clear message appears, not raw JSON
- [ ] Open Chrome DevTools → toggle device toolbar → set width to 375px — no horizontal
      scroll, input box fits, text is readable